In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic


load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
# Helper functions


def add_user_message(messages, message):
    if isinstance(message, list):
        user_message = {
            "role": "user",
            "content": message,
        }
    else:
        user_message = {
            "role": "user",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(user_message)


def add_assistant_message(messages, message):
    if isinstance(message, list):
        assistant_message = {
            "role": "assistant",
            "content": message,
        }
    elif hasattr(message, "content"):
        content_list = []
        for block in message.content:
            if block.type == "text":
                content_list.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                content_list.append(
                    {
                        "type": "tool_use",
                        "id": block.id,
                        "name": block.name,
                        "input": block.input,
                    }
                )
        assistant_message = {
            "role": "assistant",
            "content": content_list,
        }
    else:
        # String messages need to be wrapped in a list with text block
        assistant_message = {
            "role": "assistant",
            "content": [{"type": "text", "text": message}],
        }
    messages.append(assistant_message)


def chat_stream(
    messages,
    system=None,
    temperature=None,
    stop_sequences=[],
    tools=None,
    tool_choice=None,
    betas=None,
):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if temperature is not None:
        params["temperature"] = temperature 
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    if tool_choice:
        params["tool_choice"] = tool_choice

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if betas:
        params["betas"] = betas

    return client.beta.messages.stream(**params)


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
#create too ddefintion 

In [4]:
from anthropic.types import ToolParam

In [5]:
#define the tools being used including the save article schema and save short article schema
save_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence maximum",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "word_count"
                        },
                        "review": {
                            "type": "string",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Eight-sentence review of the paper",
                        },
                    },
                    "required": ["word_count", "review"]
                },
            },
            "required": ["abstract", "meta"]
        },
    }
)

save_short_article_schemna = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence maximum",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "abstract": {
                            "type": "integer",
                            "description": "Word Count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Review of paper. One short sentence maximum"
                        },
                    },
                    "required": ["word_count", "review"]
                },  
            },
            "required": ["abstract", "meta"]
        },
    }
)

def save_article(**kwargs):
    return "Article saved!"

In [10]:
#establish the code required for the tool to be running 
import json

def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)

def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False
            }

        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool.request.id,
                "content": f"Error: {e}",
                "is_error": True
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [11]:
#develop function to run the conversation 

In [12]:
def run_conversation(messages, tools=[], tool_choice=None, fine_grained=False):
    while True:
        with chat_stream(
            messages,
            tools=tools,
            betas=["fine-grained-tool-streaming-2025-05-14"] if fine_grained else [],
            tool_choice=tool_choice,
        ) as stream:
            for chunk in stream:
                if chunk.type == "text":
                    print(chunk.text, end="")

                if chunk.type == "content_block_start":
                    if chunk.content_block.type == "tool_use":
                        print(f'\n>>> Tool Call: "{chunk.content_block.name}"')

                if chunk.type == "input_json" and chunk.partial_json:
                    print(chunk.partial_json, end="")

                if chunk.type == "contnet_block_stop":
                    print("\n")

            response = stream.get_final_message()

        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        if tool_choice:
            break

    return messages 

In [13]:
messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article",
)

run_conversation(
    messages,
    tools=[save_article_schema]
)

C:\Users\alexh\AppData\Local\Temp\ipykernel_47752\2587184180.py:3: DeprecationWarning: The model 'claude-sonnet-4-5' is deprecated and will reach end-of-life on November 30th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  with chat_stream(


I'll create and save a fake computer science article for you.
>>> Tool Call: "save_article"
{"abstract": "This paper introduces a novel quantum-resistant blockchain consensus algorithm that achieves O(log n) communication complexity while maintaining Byzantine fault tolerance.", "meta": {"word_count":6547,"review":"This paper presents QuantumChain, an innovative consensus mechanism designed for distributed ledger systems in the post-quantum era. The authors cleverly combine lattice-based cryptography with a modified proof-of-stake protocol to achieve resistance against attacks from quantum computers. The theoretical analysis demonstrates logarithmic communication complexity, which represents a significant improvement over existing Byzantine fault-tolerant protocols. Experimental results on a network of 1000 nodes show throughput of 50,000 transactions per second with sub-second finality. However, the security proofs rely on several unproven assumptions about lattice problem hardness th

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'll create and save a fake computer science article for you."},
   {'type': 'tool_use',
    'id': 'toolu_01TX2Lv4KPe8XLHycbCxNuz8',
    'name': 'save_article',
    'input': {'abstract': 'This paper introduces a novel quantum-resistant blockchain consensus algorithm that achieves O(log n) communication complexity while maintaining Byzantine fault tolerance.',
     'meta': {'word_count': 6547,
      'review': 'This paper presents QuantumChain, an innovative consensus mechanism designed for distributed ledger systems in the post-quantum era. The authors cleverly combine lattice-based cryptography with a modified proof-of-stake protocol to achieve resistance against attacks from quantum computers. The theoretical analysis demonstrates logarithmic communication complexity, which represents a significant improve